In [ ]:
# print(QUOTATION_SEGMENT_DEFINITION.doc_type)
# print(QUOTATION_SEGMENT_DEFINITION.aliases)
# for s in QUOTATION_SEGMENT_DEFINITION.segments:
#     print(s.name)
#     print(s.description)
#     print("+++++++++++++++")
#     print("+++++++++++++++", s.name)

#     for f in s.fields:
#         print("=========", f.name)
#         print(f.description)
#         print(f.keyword_def)
#         print(f.keyword_def.keywords)
#         print(f.keyword_def.boost_score)
#         print()


seg

In [ ]:
# LayoutSegmentMappingRecommendation에서 세그먼트별 데이터 추출
def get_segment_data(recommendation, blocks):
    """segment_buckets 기준으로 세그먼트별 블록 데이터를 dict로 반환. { segment_name: [block, ...] }"""
    blocks_by_ref = {
        getattr(b, "ref", None): b for b in blocks if getattr(b, "ref", None)
    }
    segment_data = {}
    for bucket in recommendation.segment_buckets:
        segment_data[bucket.segment_name] = [
            blocks_by_ref[ref] for ref in bucket.block_refs if ref in blocks_by_ref
        ]
    # 미매핑 블록도 함께 반환 (선택)
    segment_data["_unmapped"] = [
        blocks_by_ref[ref]
        for ref in recommendation.unmapped_block_refs
        if ref in blocks_by_ref
    ]
    return segment_data


segment_data = get_segment_data(seg, blocks)

for name, blks in segment_data.items():
    print(f"=== {name} ({len(blks)} blocks) ===")
    for b in blks:
        ct = getattr(b, "content_type", "")
        content = getattr(b, "content", "")
        if ct == "text":
            print(f"  {b.ref}:")
            print(f"    {content}")
        elif ct == "group_kv" and isinstance(content, dict):
            print(f"  {b.ref}:")
            # print(type(content))
            # for i in content.items():
            print(f"    ", content)
        elif ct == "table" and isinstance(content, list):
            print(f"  {b.ref}: (rows={len(content)})")
            for i, row in enumerate(content):
                print(f"    {row}")
        else:
            print(f"  {b.ref}: ({ct}) {content}")
    print()

In [ ]:
from document_parsing_engine.domain.presets import QUOTATION_SEGMENT_DEFINITION

for s in QUOTATION_SEGMENT_DEFINITION.segments:

    print("+++++++++++++++")
    print("+++++++++++++++", s.name)

    for f in s.fields:
        print("====", f.name)
        print("====", f.description)
        print("====", f.keyword_def.keywords)
        print()
        if f.children:
            for c in f.children:
                print("=========", c.name)
                print("=========", c.description)
                print("=========", c.keyword_def.keywords)
                print()
    print()
    print()

In [ ]:
from document_parsing_engine.domain.presets.segment_definitions import (
    get_segment_definition,
)

seg_def = get_segment_definition(classification_result.doc_type.value)  # 'quotation'
seg_def.doc_type
# get_segment_definition("purchase_order").doc_type  # 'purchase_order'

In [ ]:
# QUOTATION_SEGMENT_DEFINITION.segments → 세그먼트별 필드 키워드 매핑 { segment_name: { field_name: [keywords, ...] } }
from document_parsing_engine.domain.presets.segment_definitions import (
    QUOTATION_SEGMENT_DEFINITION,
)


def build_segment_field_keywords(segments):
    """세그먼트 정의에서 세그먼트별·필드별 키워드 매핑. 자식 없음: field -> [keywords]; 자식 있음: field -> {"keywords": [...], child_name: [...]}."""
    out = {}
    for seg in segments:
        out[seg.name] = {}
        for f in seg.fields:
            kw = list(getattr(f.keyword_def, "keywords", ()) or ())
            if not f.children:
                out[seg.name][f.name] = kw
            else:
                out[seg.name][f.name] = {"keywords": kw}
                for c in f.children:
                    out[seg.name][f.name][c.name] = list(
                        getattr(c.keyword_def, "keywords", ()) or ()
                    )
    return out


segment_field_keywords = build_segment_field_keywords(seg_def.segments)
segment_field_keywords

In [ ]:
classification_result.doc_type.value

In [ ]:
segment_data

In [ ]:
# segment_data → JSON (BlockContent를 dict로 변환 후 직렬화)
import json


def segment_data_to_json_ready(segment_data):
    """segment_data(세그먼트별 BlockContent 리스트)를 json.dumps 가능한 dict로 변환."""
    out = {}
    for name, blocks in segment_data.items():
        out[name] = []
        for b in blocks:
            out[name].append(
                {
                    "ref": getattr(b, "ref", None),
                    "label": getattr(b, "label", None),
                    "content_type": getattr(b, "content_type", None),
                    "content": getattr(b, "content", None),
                }
            )
    return out


segment_data_json = segment_data_to_json_ready(segment_data)
segment_data_json_str = json.dumps(segment_data_json, ensure_ascii=False, indent=2)
# 파일 저장: Path("segment_data.json").write_text(segment_data_json_str, encoding="utf-8")
segment_data_json

In [ ]:
from anthropic import Anthropic
import time
from typing import Any

import os
api_key = os.environ.get("ANTHROPIC_API_KEY", "")
client = Anthropic(api_key=api_key)


model_mapping = {"claude-sonnet": "claude-sonnet-4-6"}
actual_model = model_mapping.get("claude-sonnet")

start_time = time.time()
message = client.messages.create(
    model=actual_model,
    max_tokens=50,
    messages=[{"role": "user", "content": "안녕하세요! 간단한 연결 테스트입니다."}],
)
response_time = time.time() - start_time

test_response_text = (
    message.content[0].text[:100] + "..."
    if message.content
    and len(message.content) > 0
    and len(message.content[0].text) > 100
    else (
        message.content[0].text if message.content and len(message.content) > 0 else ""
    )
)

test_response_text

In [ ]:
# 프롬프트 넣어서 호출 (위쪽 셀에서 client, actual_model 사용)
def call_with_prompt(
    prompt: str,
    model: str | None = None,
    max_tokens: int = 1024,
    client_instance=None,
) -> str:
    """프롬프트 문자열을 넣으면 Claude API를 호출하고 응답 텍스트를 반환."""
    c = client_instance or client
    m = model or actual_model
    msg = c.messages.create(
        model=m,
        max_tokens=max_tokens,
        messages=[{"role": "user", "content": prompt}],
    )
    if msg.content and len(msg.content) > 0:
        return msg.content[0].text
    return ""


# extraction_prompts.yaml 에서 시스템 + 유저 프롬프트 로드 → 합쳐서 출력 후 API 호출
import json
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))
from document_parsing_engine.prompt import get_extraction_prompts

# 변수: segment_field_mapping_metadata, segment_mapping_result_data
try:
    _ = segment_data_json_str
except NameError:
    _sd = segment_data if "segment_data" in dir() else {}
    _json = {
        n: [
            {
                "ref": getattr(b, "ref", None),
                "label": getattr(b, "label", None),
                "content_type": getattr(b, "content_type", None),
                "content": getattr(b, "content", None),
            }
            for b in blks
        ]
        for n, blks in _sd.items()
    }
    segment_data_json_str = json.dumps(_json, ensure_ascii=False, indent=2)

variables = {
    "segment_field_mapping_metadata": (
        json.dumps(segment_field_keywords, ensure_ascii=False, indent=2)
        if "segment_field_keywords" in dir()
        else "{}"
    ),
    "segment_mapping_result_data": (
        segment_data_json_str
        if "segment_data_json_str" in dir()
        else json.dumps({}, ensure_ascii=False)
    ),
}
system_prompt, user_prompt = get_extraction_prompts(variables)

# 시스템 + 유저 프롬프트 합쳐서 출력
combined = (
    "=== 시스템 프롬프트 ===\n\n"
    + system_prompt
    + "\n\n=== 유저 프롬프트 ===\n\n"
    + user_prompt
)
print(combined)

In [ ]:
# (선택) API 호출: system + user 로 전달
msg = client.messages.create(
    model=actual_model,
    max_tokens=4096,
    system=system_prompt,
    messages=[{"role": "user", "content": user_prompt}],
)
response_text = msg.content[0].text if msg.content else ""
print("\n=== API 응답 ===\n")
print(response_text)
# response_text

In [ ]:
print(response_text)